# Notebook 01: Exploração de Dados Demográficos — IBGE

**Projeto:** `migracao-venezuelana-oeste-sc`  
**Autores:** Leonardo Rafael Santos Leitão e Vicente Neves da Silva Ribeiro (UFFS)  
**Data:** Abril de 2026

---

## Objetivo

Este notebook tem como propósito explorar dados demográficos do IBGE para o Oeste de Santa Catarina, com foco na população venezuelana. As análises incluem:

1. Carregamento e inspeção dos dados;
2. Construção de pirâmides etárias;
3. Cálculo de indicadores demográficos básicos;
4. Visualizações comparativas entre população venezuelana e total.

## Fontes de Dados

- **Censo Demográfico 2022** (IBGE): microdados de pessoas por país de nascimento.
- **PNAD Contínua** (2018–2024): estimativas anuais de população.
- **Malhas municipais** (IBGE): para visualização espacial.

> **Nota metodológica:** Os dados apresentados neste notebook são **placeholder/simulados** para fins de estruturação do pipeline. Os dados reais serão obtidos via API SIDRA e microdados do IBGE.


## 1. Configuração do Ambiente

Importação das bibliotecas necessárias e configuração de estilos visuais.

In [ ]:
# -----------------------------------------------------------------------------
# Bibliotecas padrão
# -----------------------------------------------------------------------------
import os
import sys
from pathlib import Path
from typing import Optional

# -----------------------------------------------------------------------------
# Bibliotecas científicas e de dados
# -----------------------------------------------------------------------------
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# Bibliotecas de visualização
# -----------------------------------------------------------------------------
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Tentativa de importar plotly (opcional)
try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("[AVISO] plotly não instalado. Visualizações interativas desabilitadas.")

# -----------------------------------------------------------------------------
# Configurações globais
# -----------------------------------------------------------------------------
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

# Diretórios do projeto
PROJECT_ROOT = Path(".").resolve().parent
DATA_DIR = PROJECT_ROOT / "data"
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Diretório do projeto: {PROJECT_ROOT}")
print(f"Diretório de figuras: {FIGURES_DIR}")
print(f"Plotly disponível: {HAS_PLOTLY}")

## 2. Carregamento da Configuração

Carregamos parâmetros do projeto a partir de um arquivo de configuração (YAML/JSON) ou de um dicionário Python. Isso garante reprodutibilidade e centralização de parâmetros.

In [ ]:
# -----------------------------------------------------------------------------
# Configuração do projeto (placeholder até criação do config.yaml)
# -----------------------------------------------------------------------------
CONFIG = {
    "projeto": "migracao-venezuelana-oeste-sc",
    "ano_censo": 2022,
    "periodo_pnadc": [2018, 2019, 2020, 2021, 2022, 2023, 2024],
    "ufs_alvo": ["SC"],
    "mesorregiao": "Oeste Catarinense",
    "codigos_municipios_oeste_sc": [
        4204202,  # Chapecó
        4219407,  # Xanxerê
        4204301,  # Concórdia
        4209003,  # Joaçaba
        4217204,  # São Miguel do Oeste
        # TODO: incluir os demais 27 municípios
    ],
    "cores": {
        "venezuelana_homem": "#1f77b4",
        "venezuelana_mulher": "#ff7f0e",
        "total_homem": "#aec7e8",
        "total_mulher": "#ffbb78",
    },
}

print("Configuração carregada com sucesso.")
print(f"Ano do Censo: {CONFIG['ano_censo']}")
print(f"Municípios no recorte: {len(CONFIG['codigos_municipios_oeste_sc'])} (parcial)")

## 3. Carregamento dos Dados (Placeholder/Simulado)

Como os microdados do IBGE ainda não foram obtidos, geramos um conjunto de dados sintéticos que reproduzem a estrutura esperada. Esta célula será substituída pela leitura real dos arquivos CSV/Parquet.

In [ ]:
# -----------------------------------------------------------------------------
# Geração de dados sintéticos para demonstração
# -----------------------------------------------------------------------------
np.random.seed(42)

n_venezuelanos = 1_200
n_total = 50_000

# População venezuelana: perfil jovem, masculino predominante
idade_ven = np.concatenate([
    np.random.normal(loc=28, scale=8, size=int(n_venezuelanos * 0.7)).astype(int),
    np.random.normal(loc=10, scale=5, size=int(n_venezuelanos * 0.2)).astype(int),
    np.random.normal(loc=55, scale=10, size=int(n_venezuelanos * 0.1)).astype(int),
])
idade_ven = np.clip(idade_ven, 0, 100)
sexo_ven = np.random.choice([1, 2], size=n_venezuelanos, p=[0.62, 0.38])

df_ven = pd.DataFrame({
    "idade": idade_ven,
    "sexo": sexo_ven,
    "nacionalidade": "Venezuelana",
    "municipio": np.random.choice(CONFIG["codigos_municipios_oeste_sc"], size=n_venezuelanos),
    "ano": CONFIG["ano_censo"],
})

# População total do município: perfil mais equilibrado
idade_total = np.concatenate([
    np.random.normal(loc=35, scale=20, size=int(n_total * 0.8)).astype(int),
    np.random.normal(loc=5, scale=3, size=int(n_total * 0.2)).astype(int),
])
idade_total = np.clip(idade_total, 0, 100)
sexo_total = np.random.choice([1, 2], size=n_total, p=[0.495, 0.505])

df_total = pd.DataFrame({
    "idade": idade_total,
    "sexo": sexo_total,
    "nacionalidade": "Brasileira",
    "municipio": np.random.choice(CONFIG["codigos_municipios_oeste_sc"], size=n_total),
    "ano": CONFIG["ano_censo"],
})

df = pd.concat([df_ven, df_total], ignore_index=True)
df["faixa_etaria"] = pd.cut(
    df["idade"],
    bins=[0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 100],
    labels=["0-4", "5-9", "10-14", "15-19", "20-24", "25-29", "30-34",
            "35-39", "40-44", "45-49", "50-54", "55-59", "60-64",
            "65-69", "70-74", "75-79", "80+"],
    right=False,
)

print(f"Dimensões do DataFrame: {df.shape}")
print("\nAmostra dos dados:")
display(df.head(10))
print("\nEstatísticas descritivas:")
display(df.describe())

## 4. Pirâmide Etária

Construímos pirâmides etárias comparativas entre a população venezuelana e a população total do Oeste de SC. A pirâmide é uma ferramenta clássica da demografia para visualizar a estrutura etária de uma população.

In [ ]:
# -----------------------------------------------------------------------------
# Preparação dos dados para pirâmide etária
# -----------------------------------------------------------------------------
def prepara_piramide(df: pd.DataFrame, grupo: str) -> pd.DataFrame:
    """
    Agrupa dados por faixa etária e sexo, calculando percentuais.
    """
    sub = df[df["nacionalidade"] == grupo] if grupo != "Total" else df
    agg = sub.groupby(["faixa_etaria", "sexo"]).size().reset_index(name="contagem")
    total = sub.shape[0]
    agg["pct"] = (agg["contagem"] / total) * 100
    # Sexo 1 = Masculino (negativo para plotagem à esquerda)
    agg["pct_plot"] = agg.apply(lambda r: -r["pct"] if r["sexo"] == 1 else r["pct"], axis=1)
    agg["sexo_label"] = agg["sexo"].map({1: "Homem", 2: "Mulher"})
    return agg

piramide_ven = prepara_piramide(df, "Venezuelana")
piramide_total = prepara_piramide(df, "Brasileira")

In [ ]:
# -----------------------------------------------------------------------------
# Visualização: Pirâmide Etária (Matplotlib)
# -----------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharey=True)

def plot_piramide(ax, data, title, color_m, color_f):
    faixas = data["faixa_etaria"].cat.categories
    y = np.arange(len(faixas))
    homens = data[data["sexo"] == 1].set_index("faixa_etaria")["pct_plot"].reindex(faixas).fillna(0)
    mulheres = data[data["sexo"] == 2].set_index("faixa_etaria")["pct_plot"].reindex(faixas).fillna(0)

    ax.barh(y, homens, color=color_m, label="Homem", edgecolor="white", height=0.8)
    ax.barh(y, mulheres, color=color_f, label="Mulher", edgecolor="white", height=0.8)

    ax.set_yticks(y)
    ax.set_yticklabels(faixas)
    ax.set_xlabel("Percentual da população (%)")
    ax.set_title(title, fontweight="bold", fontsize=14)
    ax.legend(loc="lower right")
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlim(-12, 12)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{abs(x):.1f}"))
    ax.invert_yaxis()
    sns.despine(left=True, bottom=True)

plot_piramide(
    axes[0], piramide_ven,
    "População Venezuelana\nOeste de SC (simulado)",
    CONFIG["cores"]["venezuelana_homem"],
    CONFIG["cores"]["venezuelana_mulher"],
)

plot_piramide(
    axes[1], piramide_total,
    "População Brasileira\nOeste de SC (simulado)",
    CONFIG["cores"]["total_homem"],
    CONFIG["cores"]["total_mulher"],
)

fig.suptitle("Pirâmides Etárias Comparativas — Censo 2022 (Placeholder)", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()

# Salvamento da figura
fig_path = FIGURES_DIR / "01_piramide_etaria_comparativa.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
print(f"Figura salva em: {fig_path}")
plt.show()

## 5. Visualização Interativa com Plotly (Opcional)

Caso a biblioteca `plotly` esteja instalada, geramos uma versão interativa da pirâmide etária.

In [ ]:
if HAS_PLOTLY:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("População Venezuelana", "População Brasileira"),
        shared_yaxes=True,
    )

    for col, (data, nome, color_m, color_f) in enumerate([
        (piramide_ven, "Venezuelana", CONFIG["cores"]["venezuelana_homem"], CONFIG["cores"]["venezuelana_mulher"]),
        (piramide_total, "Brasileira", CONFIG["cores"]["total_homem"], CONFIG["cores"]["total_mulher"]),
    ], start=1):
        faixas = data["faixa_etaria"].cat.categories
        homens = data[data["sexo"] == 1].set_index("faixa_etaria")["pct"].reindex(faixas).fillna(0)
        mulheres = data[data["sexo"] == 2].set_index("faixa_etaria")["pct"].reindex(faixas).fillna(0)

        fig.add_trace(
            go.Bar(y=faixas, x=-homens, name=f"Homem {nome}", marker_color=color_m, orientation="h"),
            row=1, col=col,
        )
        fig.add_trace(
            go.Bar(y=faixas, x=mulheres, name=f"Mulher {nome}", marker_color=color_f, orientation="h"),
            row=1, col=col,
        )

    fig.update_layout(
        title_text="Pirâmides Etárias Interativas — Oeste de SC (Placeholder)",
        barmode="relative",
        height=600,
        width=1200,
    )
    fig.show()
else:
    print("Plotly não disponível. Pular visualização interativa.")

## 6. Indicadores Demográficos Básicos

Calculamos indicadores clássicos da demografia para a população venezuelana: razão de sexos, taxa de dependência e proporção de crianças/jovens/idosos.

In [ ]:
# -----------------------------------------------------------------------------
# Cálculo de indicadores demográficos
# -----------------------------------------------------------------------------
def calc_indicadores(df_sub: pd.DataFrame) -> dict:
    """
    Calcula indicadores demográficos básicos para um subconjunto.
    """
    total = len(df_sub)
    homens = (df_sub["sexo"] == 1).sum()
    mulheres = (df_sub["sexo"] == 2).sum()

    criancas = ((df_sub["idade"] >= 0) & (df_sub["idade"] <= 14)).sum()
    jovens = ((df_sub["idade"] >= 15) & (df_sub["idade"] <= 29)).sum()
    adultos = ((df_sub["idade"] >= 30) & (df_sub["idade"] <= 59)).sum()
    idosos = (df_sub["idade"] >= 60).sum()

    # Razão de sexos: homens por 100 mulheres
    razao_sexos = (homens / mulheres) * 100 if mulheres > 0 else np.nan

    # Taxa de dependência: (0-14 + 60+) / (15-59) * 100
    ativos = ((df_sub["idade"] >= 15) & (df_sub["idade"] <= 59)).sum()
    taxa_dependencia = ((criancas + idosos) / ativos) * 100 if ativos > 0 else np.nan

    return {
        "total": total,
        "homens": homens,
        "mulheres": mulheres,
        "razao_sexos": razao_sexos,
        "pct_criancas": (criancas / total) * 100,
        "pct_jovens": (jovens / total) * 100,
        "pct_adultos": (adultos / total) * 100,
        "pct_idosos": (idosos / total) * 100,
        "taxa_dependencia": taxa_dependencia,
        "idade_media": df_sub["idade"].mean(),
        "idade_mediana": df_sub["idade"].median(),
    }

ind_ven = calc_indicadores(df[df["nacionalidade"] == "Venezuelana"])
ind_total = calc_indicadores(df[df["nacionalidade"] == "Brasileira"])

indicadores = pd.DataFrame([ind_ven, ind_total], index=["Venezuelana", "Brasileira"]).T
indicadores = indicadores.round(2)
display(indicadores)

## 7. Salvamento de Resultados

Salvamos os dados agregados e as figuras geradas para uso em relatórios e no dashboard.

In [ ]:
# -----------------------------------------------------------------------------
# Exportação dos dados agregados
# -----------------------------------------------------------------------------
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Pirâmide etária agregada
piramide_export = pd.concat([
    piramide_ven.assign(nacionalidade="Venezuelana"),
    piramide_total.assign(nacionalidade="Brasileira"),
])
piramide_path = DATA_DIR / "gold" / "piramide_etaria_oeste_sc.csv"
piramide_path.parent.mkdir(parents=True, exist_ok=True)
piramide_export.to_csv(piramide_path, index=False)
print(f"Pirâmide etária exportada: {piramide_path}")

# Indicadores demográficos
indicadores_path = DATA_DIR / "gold" / "indicadores_demograficos.csv"
indicadores.to_csv(indicadores_path)
print(f"Indicadores exportados: {indicadores_path}")

print("\n--- Execução concluída com sucesso ---")

## 8. Próximos Passos

- [ ] Substituir dados sintéticos por microdados reais do IBGE (SIDRA/API);
- [ ] Incluir todos os 32 municípios do Oeste de SC;
- [ ] Calcular estimativas intercensitárias via projeções do IBGE;
- [ ] Cruzar com dados de naturalização (Ministério da Justiça);
- [ ] Validar resultados com estatísticas oficiais da OIM.

---

*Notebook elaborado no âmbito do projeto `migracao-venezuelana-oeste-sc`. Dados sintéticos utilizados apenas para demonstração da estrutura analítica.*